# Graph Convolutions & GraphSAGE

Companion notebook for the [Graph Convolutions lesson](https://ml-viz-ruby.vercel.app/courses/graph-neural-networks/02-graph-convolutions).

We implement a **GCN layer** from scratch in NumPy — including the symmetric normalization
`Â = D̃^{-1/2}(A+I)D̃^{-1/2}` — run a 2-layer forward pass, and compare GCN's full-neighborhood
aggregation to **GraphSAGE-style neighbor sampling**. Pure NumPy.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(0)

## Intuition — message passing with learned weights

A **GCN layer** upgrades plain message passing in two ways: normalize the adjacency **symmetrically**
(`Â = D̃^{-1/2}(A+I)D̃^{-1/2}`, so hub neighbors don't dominate and the operation is spectrally
well-behaved), and mix in a **learned weight matrix**: one layer is `σ(Â H W)`. Stack two and every
node's prediction uses its 2-hop neighborhood. **GraphSAGE** makes this scale to huge graphs by
*sampling* a fixed number of neighbors instead of aggregating all of them. We implement both, trace the
lesson's worked example, and verify the normalization's key spectral property.

## 1 — The normalized adjacency Â

Add self-loops, then symmetrically normalize by degree. The √ on both sides down-weights
high-degree neighbors (a hub shouldn't dominate).

In [ ]:
def normalized_adjacency(A):
    A_tilde = A + np.eye(A.shape[0])          # add self-loops
    d = A_tilde.sum(axis=1)                    # degrees
    D_inv_sqrt = np.diag(1.0 / np.sqrt(d))
    return D_inv_sqrt @ A_tilde @ D_inv_sqrt   # symmetric normalization

A = np.zeros((5, 5))
for u, v in [(0,1),(1,2),(2,3),(3,4),(1,3)]:
    A[u,v] = A[v,u] = 1
A_hat = normalized_adjacency(A)
print('Â =\n', A_hat)
print('row sums (not 1 — symmetric, not row-stochastic):', A_hat.sum(1))

**What to notice:** the symmetric normalization divides each edge weight by `√(dᵤ·dᵥ)` — an edge to
a high-degree hub counts for less. Rows *don't* sum to 1 (it's not a random walk); the symmetry is
deliberate, because it keeps `Â`'s eigenvalues in a safe range (verified below).

### The lesson's 3-node worked example

Reproduce the path graph $1\!-\!2\!-\!3$ trace from the lesson: self-loop entries
$0.5, 0.333, 0.5$, edge entries $1/\sqrt{6} \approx 0.408$, and one aggregation of
$\mathbf{h} = [1, 0, -1]$ giving $[0.5, 0, -0.5]$.

In [ ]:
A_path = np.array([[0,1,0],[1,0,1],[0,1,0]], float)
A_hat_path = normalized_adjacency(A_path)
h = np.array([1.0, 0.0, -1.0])
print('Â =\n', A_hat_path.round(3))
print('one aggregation Âh =', (A_hat_path @ h).round(3))   # → [ 0.5  0. -0.5]
print('one more layer Â(Âh) =', (A_hat_path @ (A_hat_path @ h)).round(3))


**What to notice:** the 3-node path reproduces the lesson's numbers exactly — self-loop entries
0.5/0.333/0.5, edges ≈ 0.408, and aggregating `[1, 0, −1]` gives `[0.5, 0, −0.5]`: each node moved
toward its neighbors while the symmetric structure preserved the sign pattern.

## The library way — verify the spectral property that makes GCN stable

The reason for the `D^{-1/2}` on *both* sides: it bounds `Â`'s eigenvalues to `(−1, 1]`, so stacking
layers can neither explode nor oscillate. Check it with `numpy.linalg`, and contrast the
**unnormalized** `A+I`, whose largest eigenvalue exceeds 1 and grows with degree.

In [ ]:
eigs_hat = np.linalg.eigvalsh(A_hat)
eigs_raw = np.linalg.eigvalsh(A + np.eye(5))
print('eigenvalues of Â         :', eigs_hat.round(3), ' (max =', eigs_hat.max().round(3), ')')
print('eigenvalues of A+I (raw) :', eigs_raw.round(3), ' (max =', eigs_raw.max().round(3), ')')
assert eigs_hat.max() <= 1 + 1e-9 and eigs_hat.min() > -1 - 1e-9, "Â's spectrum must lie in (-1, 1]"
assert eigs_raw.max() > 1.5, "the unnormalized operator amplifies signals"
K = 10
Xs = rng.normal(size=(5, 4))                 # a random signal to propagate
print(f'\nnorm of Â^{K} @ Xs   : {np.linalg.norm(np.linalg.matrix_power(A_hat, K) @ Xs):8.2f}  (stable)')
print(f'norm of (A+I)^{K} @ Xs: {np.linalg.norm(np.linalg.matrix_power(A + np.eye(5), K) @ Xs):8.2f}  (explodes)')

**What to notice:** `Â`'s spectrum sits inside `(−1, 1]` so ten stacked applications keep the signal
norm tame, while the raw `A+I` (top eigenvalue ~3.4) blows the norm up by orders of magnitude. The
normalization is the GCN's stability guarantee — the same spectral-radius logic as RNN gradients and
weight init, appearing again.

## 2 — A GCN layer and a 2-layer forward pass

One layer is `σ(Â H W)`. We stack two layers (hidden ReLU, then a linear output) — a complete GCN
forward pass for, say, 3-class node classification.

In [ ]:
def relu(x):
    return np.maximum(x, 0)

def gcn_layer(A_hat, H, W):
    return A_hat @ H @ W

n, d_in, d_hid, d_out = 5, 4, 8, 3
X = rng.normal(size=(n, d_in))                 # node features
W1 = rng.normal(size=(d_in, d_hid)) * 0.5
W2 = rng.normal(size=(d_hid, d_out)) * 0.5

H1 = relu(gcn_layer(A_hat, X, W1))             # layer 1 (2-hop info after layer 2)
logits = gcn_layer(A_hat, H1, W2)              # layer 2 -> class logits per node
probs = np.exp(logits) / np.exp(logits).sum(1, keepdims=True)
print('per-node class probabilities:\n', probs)
print('predicted classes:', probs.argmax(1))

**What to notice:** the full 2-layer GCN is four lines: `relu(Â X W₁)` then `Â H W₂` → per-node
class logits. After two layers, every node's prediction mixes features from its 2-hop neighborhood —
receptive field = number of layers, exactly like CNNs.

## 3 — GraphSAGE: sampling neighbors

On a huge graph you can't aggregate every neighbor of a hub. GraphSAGE samples a fixed number per
node. We compare the full mean aggregation to a sampled estimate — sampling approximates the full
aggregation while bounding cost regardless of degree.

In [ ]:
def full_mean(A, X, v):
    nb = np.where(A[v] > 0)[0]
    return X[nb].mean(0)

def sampled_mean(A, X, v, k, rng):
    nb = np.where(A[v] > 0)[0]
    take = rng.choice(nb, size=min(k, len(nb)), replace=False)
    return X[take].mean(0)

# make node 1 a hub with many neighbors
Abig = A.copy()
Xbig = rng.normal(size=(5, 4))
node = 1
full = full_mean(Abig, Xbig, node)
samp = np.mean([sampled_mean(Abig, Xbig, node, k=2, rng=np.random.default_rng(s)) for s in range(200)], axis=0)
print('full neighborhood mean:   ', full)
print('avg of sampled estimates: ', samp)
print('sampling is an unbiased estimate of the full mean ->', np.allclose(full, samp, atol=0.05))

**What to notice:** the sampled mean tracks the full mean with error shrinking as the sample grows —
Monte Carlo again. GraphSAGE's insight is that a hub with 10,000 neighbors doesn't need all of them:
a fixed sample (say 25) bounds the cost per node *regardless of degree*, making minibatch training on
billion-edge graphs possible.

## Gotchas & tradeoffs

- **Normalization isn't optional** — unnormalized aggregation explodes with depth and lets hubs
  dominate (shown above).
- **GCN is transductive by default** (needs the whole graph's `Â`); GraphSAGE's sample-and-aggregate is
  **inductive** — it generalizes to unseen nodes/graphs.
- **Sampling adds variance:** smaller neighbor samples are cheaper but noisier — the usual Monte-Carlo
  tradeoff.
- **Depth still over-smooths:** normalization fixes explosion, not homogenization; 2–3 layers remains
  the sweet spot.

In [ ]:
# Sampling variance: estimate quality vs neighbors sampled (hub with 50 neighbors)
A_big = np.zeros((51, 51)); A_big[0, 1:] = A_big[1:, 0] = 1; A_big += np.eye(51)
X_big = rng.normal(size=(51, 4))
full = X_big[np.where(A_big[0] > 0)[0]].mean(0)
for k in [5, 15, 40]:
    errs = [np.linalg.norm(sampled_mean(A_big, X_big, 0, k, np.random.default_rng(s)) - full)
            for s in range(200)]
    print(f'sample {k:>2} of 51 neighbors: mean error = {np.mean(errs):.3f}')

**What to notice:** the sampled aggregation's error falls as the sample grows — `1/√k` Monte-Carlo
behavior. GraphSAGE chooses `k` to balance accuracy against a *hard* per-node compute bound; that
bound, not accuracy, is what makes web-scale GNNs feasible.

## ✏️ Your turn

**Exercise.** Implement `gcn_forward(A, X, W1, W2)` doing a full 2-layer GCN forward pass: build the
normalized adjacency, apply `relu(Â X W1)`, then `Â H1 W2`, and return the logits. Reuse
`normalized_adjacency`, `relu`, and `gcn_layer` above.

In [ ]:
def gcn_forward(A, X, W1, W2):
    # TODO(you): normalize A, run two GCN layers (ReLU after the first), return logits
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
out = gcn_forward(A, X, W1, W2)
assert out.shape == (5, 3), out.shape
assert np.allclose(out, logits)            # matches the step-by-step pass above
# symmetric normalization is symmetric
assert np.allclose(A_hat, A_hat.T)
print('\u2713 GCN forward pass is correct')

<details>
<summary>Solution</summary>

```python
def gcn_forward(A, X, W1, W2):
    A_hat = normalized_adjacency(A)
    H1 = relu(gcn_layer(A_hat, X, W1))
    return gcn_layer(A_hat, H1, W2)
```

Two layers give each node a 2-hop receptive field. The shared weights W1, W2 are the learnable
'filters' — trained by backprop against node labels, exactly like a CNN, but over graph neighborhoods
instead of pixel grids.

</details>